# Phase 3: Generative AI (GAN) Implementation

This notebook implements a Deep Convolutional GAN (DCGAN) to generate synthetic fabric defect images.

**Goals:**
1. Define the Generator and Discriminator architectures.
2. Train the GAN exclusively on the `defect` class images.
3. Generate and save synthetic images to `synthetic_defect/` for later use in Phase 4.

**Key improvements over baseline DCGAN:**
- Stronger Generator (ngf=128) vs weaker Discriminator (ndf=64)
- Data augmentation to prevent GAN memorization on small dataset
- Spectral Normalization + Dropout in Discriminator for regularization
- One-sided label smoothing (real_label=0.9)
- Instance noise on real images fed to Discriminator
- Learning rate scheduling

In [ ]:
# 1. Imports & Setup
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as dset
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
import os

# Constants
IMG_SIZE = 128
batch_size = 32  # Smaller batch = more updates per epoch with small dataset
nz = 100         # Size of z latent vector (i.e. size of generator input)
ngf = 128        # Generator feature maps (INCREASED for stronger G)
ndf = 64         # Discriminator feature maps (kept smaller to weaken D)
num_epochs = 200 # GANs need many epochs
lr = 0.0002
beta1 = 0.5
ngpu = 1

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")
print(f"Device: {device}")

In [ ]:
# 2. Load Defect Data Only (with Data Augmentation)
# We strictly only want defect images

from torch.utils.data import Dataset
from PIL import Image
import glob

class SingleClassDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = glob.glob(os.path.join(root_dir, '*.png')) + glob.glob(os.path.join(root_dir, '*.jpg'))
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, 0 # Label doesn't matter for GAN

# Data Augmentation: crucial for small datasets to prevent GAN memorization
gan_transforms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

gan_dataset = SingleClassDataset(root_dir='processed_data/train_defect',
                                 transform=gan_transforms)

dataloader = torch.utils.data.DataLoader(gan_dataset, batch_size=batch_size,
                                         shuffle=True, num_workers=2, drop_last=True)

print(f"GAN Training Images: {len(gan_dataset)}")
print(f"Batches per epoch: {len(dataloader)}")

In [ ]:
# 3. Models

# Custom weights initialization called on netG and netD
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# Generator Code (STRONGER: ngf=128)
class Generator(nn.Module):
    def __init__(self, ngpu):
        super(Generator, self).__init__()
        self.ngpu = ngpu
        self.main = nn.Sequential(
            # input is Z, going into a convolution
            nn.ConvTranspose2d(nz, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            # state size. (ngf*8) x 4 x 4
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            # state size. (ngf*4) x 8 x 8
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            # state size. (ngf*2) x 16 x 16
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # state size. (ngf) x 32 x 32
            nn.ConvTranspose2d(ngf, ngf // 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf // 2),
            nn.ReLU(True),
            # state size. (ngf/2) x 64 x 64
            nn.ConvTranspose2d(ngf // 2, 3, 4, 2, 1, bias=False),
            nn.Tanh()
            # state size. 3 x 128 x 128
        )

    def forward(self, input):
        return self.main(input)

# Discriminator Code (REGULARIZED: spectral norm + dropout)
class Discriminator(nn.Module):
    def __init__(self, ngpu):
        super(Discriminator, self).__init__()
        self.ngpu = ngpu
        self.main = nn.Sequential(
            # input is 3 x 128 x 128
            nn.utils.spectral_norm(nn.Conv2d(3, ndf, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(0.3),
            # state size. (ndf) x 64 x 64
            nn.utils.spectral_norm(nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(0.3),
            # state size. (ndf*2) x 32 x 32
            nn.utils.spectral_norm(nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(0.3),
            # state size. (ndf*4) x 16 x 16
            nn.utils.spectral_norm(nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*8) x 8 x 8
            nn.utils.spectral_norm(nn.Conv2d(ndf * 8, ndf * 8, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            # state size. (ndf*8) x 4 x 4
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, input):
        return self.main(input)

# Create models
netG = Generator(ngpu).to(device)
netG.apply(weights_init)
print(netG)

netD = Discriminator(ngpu).to(device)
netD.apply(weights_init)
print(netD)

In [ ]:
# 4. Training Loop
criterion = nn.BCELoss()

# Create batch of latent vectors that we will use to visualize
#  the progression of the generator
fixed_noise = torch.randn(64, nz, 1, 1, device=device)

# Establish convention for real and fake labels during training
# ONE-SIDED LABEL SMOOTHING: use 0.9 instead of 1.0 for real labels
real_label = 0.9
fake_label = 0.

# Setup Adam optimizers for both G and D
# Use slightly lower LR for D to slow it down
optimizerD = optim.Adam(netD.parameters(), lr=lr * 0.5, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

# Learning rate schedulers
schedulerD = optim.lr_scheduler.StepLR(optimizerD, step_size=50, gamma=0.5)
schedulerG = optim.lr_scheduler.StepLR(optimizerG, step_size=50, gamma=0.5)

print("Starting Training Loop...")
img_list = []
G_losses = []
D_losses = []
iters = 0

for epoch in range(num_epochs):
    for i, data in enumerate(dataloader, 0):
        ############################
        # (1) Update D network: maximize log(D(x)) + log(1 - D(G(z)))
        ###########################
        ## Train with all-real batch
        netD.zero_grad()
        real_cpu = data[0].to(device)
        b_size = real_cpu.size(0)
        label = torch.full((b_size,), real_label, dtype=torch.float, device=device)

        # INSTANCE NOISE: Add decaying Gaussian noise to real images for D
        noise_strength = max(0, 0.1 * (1 - epoch / num_epochs))
        real_noisy = real_cpu + noise_strength * torch.randn_like(real_cpu)

        # Forward pass real batch through D
        output = netD(real_noisy).view(-1)
        errD_real = criterion(output, label)
        # Calculate gradients for D in backward pass
        errD_real.backward()
        D_x = output.mean().item()

        ## Train with all-fake batch
        # Generate batch of latent vectors
        noise = torch.randn(b_size, nz, 1, 1, device=device)
        # Generate fake image batch with G
        fake = netG(noise)
        label.fill_(fake_label)

        # Add noise to fake images too for D
        fake_noisy = fake.detach() + noise_strength * torch.randn_like(fake.detach())

        # Classify all fake batch with D
        output = netD(fake_noisy).view(-1)
        errD_fake = criterion(output, label)
        # Calculate gradients for D in backward pass
        errD_fake.backward()
        D_G_z1 = output.mean().item()
        # Add the gradients from the all-real and all-fake batches
        errD = errD_real + errD_fake
        # Update D
        optimizerD.step()

        ############################
        # (2) Update G network: maximize log(D(G(z)))
        ###########################
        netG.zero_grad()
        label.fill_(1.0)  # G always targets 1.0 (no smoothing for G)
        # Since we just updated D, perform another forward pass of all-fake batch through D
        output = netD(fake).view(-1)
        errG = criterion(output, label)
        # Calculate gradients for G
        errG.backward()
        D_G_z2 = output.mean().item()
        # Update G
        optimizerG.step()

        # Output training stats
        if i % 10 == 0:
            print('[%d/%d][%d/%d]\tLoss_D: %.4f\tLoss_G: %.4f\tD(x): %.4f\tD(G(z)): %.4f / %.4f'
                  % (epoch, num_epochs, i, len(dataloader),
                     errD.item(), errG.item(), D_x, D_G_z1, D_G_z2))

        G_losses.append(errG.item())
        D_losses.append(errD.item())
        
        iters += 1

    # Step the learning rate schedulers
    schedulerD.step()
    schedulerG.step()
    
    # Save images every 10 epochs for monitoring
    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            fake = netG(fixed_noise).detach().cpu()
        img_list.append(vutils.make_grid(fake, padding=2, normalize=True))
        
        # Save Model checkpoint
        torch.save(netG.state_dict(), f'generator_epoch_{epoch}.pth')
        torch.save(netD.state_dict(), f'discriminator_epoch_{epoch}.pth')
        
        # Show progress
        plt.figure(figsize=(10, 10))
        plt.axis('off')
        plt.title(f'Generated Images - Epoch {epoch+1}')
        plt.imshow(np.transpose(img_list[-1], (1, 2, 0)))
        plt.show()

print("Training Finished!")

In [ ]:
# 5. Plot Training Losses
plt.figure(figsize=(10, 5))
plt.title('Generator and Discriminator Loss During Training')
plt.plot(G_losses, label='G')
plt.plot(D_losses, label='D')
plt.xlabel('iterations')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# 6. Generate and Save Synthetic Data
output_dir = 'synthetic_defect'
os.makedirs(output_dir, exist_ok=True)

num_images_needed = 600 # Generate enough to balance dataset
# Target: ~1332 normal vs ~858 real defect + 600 synthetic = ~1458 defect

netG.eval()

count = 0
batch_gen_size = 32

print(f"Generating {num_images_needed} synthetic images to {output_dir}...")

with torch.no_grad():
    while count < num_images_needed:
        noise = torch.randn(batch_gen_size, nz, 1, 1, device=device)
        fake = netG(noise).detach().cpu()
        
        for i in range(fake.size(0)):
            if count >= num_images_needed:
                break
            
            # Denormalize to [0, 1] then [0, 255]
            img = fake[i] * 0.5 + 0.5
            vutils.save_image(img, os.path.join(output_dir, f'synth_defect_{count}.png'))
            count += 1

print(f"Generation Complete. {count} images saved to {output_dir}/")

# Show some samples
sample_noise = torch.randn(16, nz, 1, 1, device=device)
with torch.no_grad():
    sample_imgs = netG(sample_noise).detach().cpu()
grid = vutils.make_grid(sample_imgs, nrow=4, padding=2, normalize=True)
plt.figure(figsize=(8, 8))
plt.axis('off')
plt.title('Final Generated Samples')
plt.imshow(np.transpose(grid, (1, 2, 0)))
plt.show()